# 08. Tópicos en el Tiempo 

Este cuaderno evalúa la evolución temporal de la agenda editorial de Revista Anfibia (2012–presente) a partir del modelo BERTopic entrenado.

**Objetivos:**
1. Integrar la dimensión temporal (`post_date`) sin necesidad de re-entrenar el modelo de lenguaje base.
2. Calcular la dinámica de tópicos mediante `topics_over_time` (evolución del c-TF-IDF y volumen de notas).
3. Medir la estacionalidad para clasificar tópicos entre **coyunturales** (event-driven / picos) y **estructurales** (cobertura continua / evergreen).

*El modelo usado es el hibrido, pero se puede cambiar por la variables NOMBRE_MODELO y CARPETA_MODELO en la celda siguiente*

In [48]:
# 1. Parche de compatibilidad obligatorio (Numba/Coverage)
import coverage.types as ct
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

# 2. Imports generales
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from bertopic import BERTopic

warnings.filterwarnings("ignore")

# 3. Configuración de Rutas y Variables
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# VARIABLE CONFIGURABLE DE MODELO
CARPETA_MODELO = "bertopic_hibrido"
NOMBRE_MODELO = "bertopic_hibrido_model"

MODEL_PATH = PROJECT_ROOT / "outputs" / CARPETA_MODELO / NOMBRE_MODELO
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
RAW_POSTS = PROJECT_ROOT / "data" / "processed" / "posteos_anfibia.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analisis_temporal"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# 4. Carga de datos y modelo
print(f"Cargando modelo: {MODEL_PATH}")
topic_model = BERTopic.load(MODEL_PATH)

df = pd.read_csv(DATA_PROCESSED)

# Si post_date no está en el corpus lematizado, lo traemos del raw sin pisar duplicados
if "post_date" not in df.columns:
    df_fechas = pd.read_csv(RAW_POSTS, usecols=["ID", "post_date", "post_year"])
    # Nos aseguramos de traer solo lo que falte
    cols_a_traer = [c for c in ["ID", "post_date", "post_year"] if c not in df.columns or c == "ID"]
    df = df.merge(df_fechas[cols_a_traer], on="ID", how="left")

df["post_date"] = pd.to_datetime(df["post_date"], errors="coerce")

# Asignar tópicos (usa los existentes si ya están guardados para ahorrar tiempo)
if "topic" not in df.columns:
    topics, _ = topic_model.transform(df["document_text_final"].fillna("").tolist())
    df["topic"] = topics

# Filtrar registros válidos con fecha y texto
df_temp = df[df["post_date"].notna() & df["document_text_final"].notna()].sort_values("post_date").reset_index(drop=True)

print(f"Total de artículos listos para análisis temporal: {len(df_temp):,}")
print(f"Rango de fechas: {df_temp['post_date'].min().strftime('%Y-%m-%d')} a {df_temp['post_date'].max().strftime('%Y-%m-%d')}")

Cargando modelo: C:\Trabajos\anfibia-unsam\outputs\bertopic_hibrido\bertopic_hibrido_model
Total de artículos listos para análisis temporal: 3,556
Rango de fechas: 2012-05-14 a 2026-06-16


## 1. Analisis topicos en el tiempo
Calculamos la evolución de frecuencias y el desplazamiento de términos representativos en el tiempo.

In [49]:
timestamps = df_temp["post_date"].dt.to_pydatetime().tolist()
docs = df_temp["document_text_final"].tolist()

print("Calculando evolución temporal de tópicos (Topics over Time)...")
topics_over_time = topic_model.topics_over_time(
    docs=docs,
    timestamps=timestamps,
    global_tuning=True,
    evolution_tuning=True,
    nr_bins=25  # Ventanas de tiempo a lo largo del histórico
)

# Exportar tabla cruda de evolución
topics_over_time.to_csv(OUTPUT_DIR / "topics_over_time_raw.csv", index=False)
display(topics_over_time.head(10))

Calculando evolución temporal de tópicos (Topics over Time)...


,Topic,Words,Frequency,Timestamp
0,-1,"carlos, kaiser, padre, junto, velázquez",7,2012-05-09 10:13:18.781
1,0,"diario, lorenzo, hombre, contar, mundo",17,2012-05-09 10:13:18.781
2,9,"pescador, barco, marino, marinero, marisco",1,2012-05-09 10:13:18.781
3,10,"rebelión, manifestación, pueblo, marcha, fiesta",1,2012-05-09 10:13:18.781
4,-1,"hombre, junto, domador, caballo, contar",8,2012-12-06 09:45:30.760
5,0,"muerto, hombre, contar, junto, sino",9,2012-12-06 09:45:30.760
6,3,"pesticida, humo, pandemia, tóxico, peligroso",1,2012-12-06 09:45:30.760
7,7,"israel, israelí, palestina, palestino, aviv",1,2012-12-06 09:45:30.760
8,-1,"periodista, obra, contar, sino, pensar",18,2013-06-30 05:47:39.520
9,0,"político, política, candidato, tal, padre",11,2013-06-30 05:47:39.520


## 2. Métricas de Estacionalidad y Clasificación 

Para entender el comportamiento de cada tópico en el tiempo calculamos:
* **Volumen Total:** Cantidad histórica de artículos.
* **Pico Máximo:** Mayor concentración de notas en una sola ventana temporal.
* **Coeficiente de Variación ($CV = \frac{\sigma}{\mu}$):** 
  * $CV$ **alto:** Tópico **coyuntural / event-driven** (picos concentrados en momentos específicos, ej. elecciones, leyes, pandemia).
  * $CV$ **bajo:** Tópico **Atemporal / evergreen** (constante a lo largo del tiempo).

In [50]:
# 1. Filtramos el ruido (-1) para el análisis editorial
tot_validos = topics_over_time[topics_over_time["Topic"] != -1].copy()

# 2. Agrupación y cálculo estadístico por tópico
metricas_estabilidad = (
    tot_validos.groupby("Topic")
    .agg(
        volumen_total=("Frequency", "sum"),
        pico_maximo=("Frequency", "max"),
        frecuencia_media=("Frequency", "mean"),
        desvio_std=("Frequency", "std")
    )
    .reset_index()
)

# 3. EXTRAER NOMBRES GLOBALES REALES (Para que coincidan con la imagen)
info_global = topic_model.get_topic_info()
# Armamos un diccionario mapeando el ID del topico con sus top 5 palabras globales
nombres_globales = {
    row["Topic"]: ", ".join(row["Representation"][:5]) 
    for _, row in info_global.iterrows() if row["Topic"] != -1
}
# Asignamos los nombres reales
metricas_estabilidad["terminos_historicos"] = metricas_estabilidad["Topic"].map(nombres_globales)

# 4. Coeficiente de variación (llenamos con 0 si desvío es NaN por baja frecuencia)
metricas_estabilidad["desvio_std"] = metricas_estabilidad["desvio_std"].fillna(0)
metricas_estabilidad["coef_variacion"] = (
    metricas_estabilidad["desvio_std"] / metricas_estabilidad["frecuencia_media"]
).round(2)

# 5. Clasificación: usamos el percentil 60 del CV como corte empírico
umbral_coyuntural = metricas_estabilidad["coef_variacion"].quantile(0.60)

metricas_estabilidad["dinamica_temporal"] = np.where(
    metricas_estabilidad["coef_variacion"] >= umbral_coyuntural,
    "Coyuntura", #Coyuntural / Event-Driven
    "Atemporal"
)

# 6. Ordenamos por los más coyunturales (mayor CV)
metricas_estabilidad = metricas_estabilidad.sort_values(by="coef_variacion", ascending=False).reset_index(drop=True)

display(metricas_estabilidad.head(15))

,Topic,volumen_total,pico_maximo,frecuencia_media,desvio_std,terminos_historicos,coef_variacion,dinamica_temporal
0,2,86,35,5.058824,8.613003,"aborto, abortar, anticonceptivo, embarazo, emb...",1.70,Coyuntura
1,3,75,21,5.000000,6.244998,"pandemia, epidemia, vacunar, vacuna, vacunación",1.25,Coyuntura
2,4,46,9,2.421053,2.168353,"papal, catolicismo, arzobispo, católica, católico",0.90,Coyuntura
3,1,113,19,5.380952,4.421269,"noticia, sociedad, tecnológico, periodismo, te...",0.82,Coyuntura
4,11,13,6,3.250000,2.217356,"terrorista, siria, terrorismo, musulmán, arabia",0.68,Coyuntura
5,6,27,4,1.588235,0.939336,"soviética, soviético, vladimir, rusia, ucrania",0.59,Atemporal
6,8,16,3,1.142857,0.534522,"cientificismo, astrología, universo, ciencia, ...",0.47,Atemporal
7,7,19,3,1.727273,0.786245,"israel, netanyahu, israelí, palestina, palestino",0.46,Atemporal
8,5,30,3,1.764706,0.752447,"arte, art, estética, artístico, artista",0.43,Atemporal
9,0,1686,105,67.440000,26.124829,"político, política, discurso, junto, sociedad",0.39,Atemporal


Teniendolo en cuenta para la recomendacion de articulos, los artículos de tópicos coyunturales (CV > 0.65) podrian recibir una penalización por antigüedad (*time decay*), mientras que los tópicos estructurales (CV < 0.50) pueden recomendar notas históricas sin perder vigencia.

## 2.5 Inspección Cualitativa de Tópicos
Analizemos estos topicos clasificados. Mapeamos las palabras clave algorítmicas con los títulos reales de las notas.

In [51]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ==============================================================================
# 1. PREPARACIÓN ESTADÍSTICA ESTRICTA
# ==============================================================================
minimo_notas = 20
MOSTRAR_POLITICA = False

# Filtramos Tópico -1 (ruido) y Tópico 0 (el megaclúster que aplasta al resto)
topicos_limpios = metricas_estabilidad[
    (metricas_estabilidad["Topic"] != -1) 
     & 
    (metricas_estabilidad["Topic"] != 0) if MOSTRAR_POLITICA==False else (metricas_estabilidad["Topic"] != -1)
].copy()

# Regla 1: Solo tópicos con volumen decente (> minimo_notas) para que la curva tenga sentido
topicos_robustos = topicos_limpios[topicos_limpios["volumen_total"] >= minimo_notas].copy()

# Preparar dataset temporal
df_evo = df_temp[df_temp["topic"].isin(topicos_robustos["Topic"])].copy()
df_evo["post_year"] = df_evo["post_date"].dt.year
agg = df_evo.groupby(["topic", "post_year"]).size().reset_index(name="notas")

# Regla 2: Normalización por Pico Máximo (El pico histórico = 100%)
max_por_topico = agg.groupby("topic")["notas"].max().reset_index(name="max_notas")
agg = agg.merge(max_por_topico, on="topic")
agg["intensidad_relativa"] = (agg["notas"] / agg["max_notas"]) * 100

# Ampliamos a 5 tópicos por panel
top_coyuntura = topicos_robustos.sort_values("coef_variacion", ascending=False).head(5)
top_evergreen = topicos_robustos.sort_values("coef_variacion", ascending=True).head(5)

# ==============================================================================
# GRÁFICO A: EL CUADRANTE EDITORIAL (Scatter Analítico)
# ==============================================================================
fig_scatter = px.scatter(
    topicos_robustos,
    x="volumen_total",
    y="coef_variacion",
    color="dinamica_temporal",
    size="volumen_total",
    hover_name="terminos_historicos",
    text="Topic",
    color_discrete_map={"Coyuntura": "#d62728", "Atemporal": "#2ca02c"},
    title=f'<b>Cuadrante Editorial: Coyuntura vs. Estructura (Tópicos > {minimo_notas} notas), Tópico 0: {MOSTRAR_POLITICA} </b>',
    labels={"volumen_total": "Volumen Total de Notas", "coef_variacion": "Nivel de Coyuntura (Coef. Variación)"},
    height=500
)
fig_scatter.update_traces(textposition='top center')
fig_scatter.add_hline(y=umbral_coyuntural, line_dash="dash", line_color="gray", annotation_text="Umbral de Coyuntura")
fig_scatter.write_html(OUTPUT_DIR / "03A_cuadrante_editorial.html")
fig_scatter.show()



## Time Decay
El time decay es una penalizacion que se le aplica a los algoritmos de recomendación para que el contenido viejo vaya perdiendo relevancia a medida que pasa el tiempo

Al puntaje de similitud (que va de 0 a 1) se lo multiplica por un factor que se va achicando con los años.

* Nota de hace 1 mes: multiplica por `1.0` (queda intacta).
* Nota de hace 2 años: multiplica por `0.8` (pierde 20% de peso).
* Nota de hace 10 años: multiplica por `0.1` (se va al fondo de la lista).

**Para eso las dos categorias**

* Si la nota es de **coyuntura** (pandemia, un debate de ley), se penaliza el modelo de recomendación para que recomiende cosas frescas.
* Si la nota es **Atemporal** (un ensayo sobre arte, politica, etc), el *time decay* tiene que ser 0. Un ensayo cultural de 2014 es igual de válido para el lector que uno publicado ayer.


In [53]:
print("Calculando penalizaciones temporales (Time Decay)...")

# Asignamos la bandera de decaimiento basada en el umbral
metricas_estabilidad["requiere_time_decay"] = metricas_estabilidad["coef_variacion"] >= umbral_coyuntural

# Calculamos un factor matemático inicial para probar luego en el recomendador
# Ej: 0.5 reduce a la mitad el score de similitud si la nota es vieja, 1.0 no lo toca.
metricas_estabilidad["factor_penalizacion_base"] = np.where(
    metricas_estabilidad["requiere_time_decay"],
    0.5, 
    1.0  
)

# Ordenamos secuencialmente por número de tópico
metricas_estabilidad = metricas_estabilidad.sort_values(by="Topic").reset_index(drop=True)

# Exportamos la tabla final maestra
ruta_exportacion = OUTPUT_DIR / "pesos_temporales_recomendador.csv"
metricas_estabilidad.to_csv(ruta_exportacion, index=False)

print(f"Insumo exportado exitosamente en: {ruta_exportacion}")
display(metricas_estabilidad[["Topic", "terminos_historicos", "dinamica_temporal", "requiere_time_decay", "factor_penalizacion_base"]].head(10))

Calculando penalizaciones temporales (Time Decay)...
Insumo exportado exitosamente en: C:\Trabajos\anfibia-unsam\outputs\analisis_temporal\pesos_temporales_recomendador.csv


,Topic,terminos_historicos,dinamica_temporal,requiere_time_decay,factor_penalizacion_base
0,0,"político, política, discurso, junto, sociedad",Atemporal,False,1.0
1,1,"noticia, sociedad, tecnológico, periodismo, te...",Coyuntura,True,0.5
2,2,"aborto, abortar, anticonceptivo, embarazo, emb...",Coyuntura,True,0.5
3,3,"pandemia, epidemia, vacunar, vacuna, vacunación",Coyuntura,True,0.5
4,4,"papal, catolicismo, arzobispo, católica, católico",Coyuntura,True,0.5
5,5,"arte, art, estética, artístico, artista",Atemporal,False,1.0
6,6,"soviética, soviético, vladimir, rusia, ucrania",Atemporal,False,1.0
7,7,"israel, netanyahu, israelí, palestina, palestino",Atemporal,False,1.0
8,8,"cientificismo, astrología, universo, ciencia, ...",Atemporal,False,1.0
9,9,"submarino, marino, océano, naval, marítimo",Atemporal,False,1.0


Tópico 6 (Rusia/Ucrania), el Tópico 7 (Israel/Palestina) y el Tópico 9 (Submarinos/ARA San Juan)

Periodísticamente pensaria que son eventos coyunturales pero el algoritmo los mandó a "Atemporal". 
Esto pasa por una limitación matemática del Coeficiente de Variación (CV): si Anfibia publica pocos articulos sobre ellos pero durante 10 años, esa constante achata el desvío estándar.

In [55]:
# Tópicos bajo sospecha editorial
topicos_sospechosos = [6, 7, 9]

print("ANÁLISIS DE TÓPICOS DUDOSOS (Posibles Falsos Atemporales)")
print("="*80)
display(
    metricas_estabilidad[metricas_estabilidad["Topic"].isin(topicos_sospechosos)][
        ["Topic", "terminos_historicos", "volumen_total", "pico_maximo", "frecuencia_media", "coef_variacion"]
    ]
)

# Generamos la data temporal directo desde df_temp para saltear el filtro de minimo_notas
df_crudo = df_temp[df_temp["topic"].isin(topicos_sospechosos)].copy()
df_crudo["post_year"] = df_crudo["post_date"].dt.year
agg_sospechosos = df_crudo.groupby(["topic", "post_year"]).size().reset_index(name="notas")

# Joineamos con los nombres
agg_sospechosos = agg_sospechosos.merge(metricas_estabilidad[["Topic", "terminos_historicos"]], left_on="topic", right_on="Topic")
agg_sospechosos["Etiqueta"] = "T" + agg_sospechosos["topic"].astype(str) + ": " + agg_sospechosos["terminos_historicos"].str[:25] + "..."

fig_sospechosos = px.bar(
    agg_sospechosos,
    x="post_year",
    y="notas",
    color="Etiqueta",
    barmode="group",
    title="<b>Evolución Anual: Rusia (T6), Israel (T7) y Submarino (T9)</b>",
    labels={"post_year": "Año", "notas": "Cantidad de Notas publicadas"},
    height=400,
    width=900
)

fig_sospechosos.update_layout(xaxis=dict(tickmode="linear", dtick=1))
fig_sospechosos.show()

ANÁLISIS DE TÓPICOS DUDOSOS (Posibles Falsos Atemporales)


,Topic,terminos_historicos,volumen_total,pico_maximo,frecuencia_media,coef_variacion
6,6,"soviética, soviético, vladimir, rusia, ucrania",27,4,1.588235,0.59
7,7,"israel, netanyahu, israelí, palestina, palestino",19,3,1.727273,0.46
9,9,"submarino, marino, océano, naval, marítimo",15,2,1.250000,0.36


## Propocisión de una tercera categoria

-Coyuntura Pura: Picos extremos por un evento único que luego cae a cero (ej. elecciones, leyes, el caso del submarino).

-Coyuntura Recurrente: Temas latentes que se mantienen con pocos articulos pero tienen picos que se repiten en el tiempo. 

-Atemporal (Evergreen): Ensayos, cultura y notas de fondo que no dependen de la fecha de publicación.

Tasa de Años Inactivos (tasa_anos_inactivos): Utilizando la matriz con ceros, se calcula el porcentaje exacto de años en los que el tópico no tuvo actividad (un valor entre 0 y 1).


- Coyuntura: coef_variacion >= umbral_cv.

- Coyuntura Recurrente: coef_variacion >= umbral_cv y tasa_años_inactivos >= 0.30

- Atemporal: Flujo constante y estable de publicación sin picos violentos ni baches.

In [72]:
# 1. Filtramos el ruido (-1)
tot_validos = topics_over_time[topics_over_time["Topic"] != -1].copy()

# 2. Reconstruimos la matriz completa de años para calcular los ceros reales
años_unicos = sorted(df_temp["post_date"].dt.year.unique())
todos_los_topics = tot_validos["Topic"].unique()
grid_completa = pd.MultiIndex.from_product([todos_los_topics, años_unicos], names=["Topic", "post_year"]).to_frame(index=False)

df_evo_anual = tot_validos.copy()
df_evo_anual["post_year"] = pd.to_datetime(df_evo_anual["Timestamp"]).dt.year
conteo_real = df_evo_anual.groupby(["Topic", "post_year"])["Frequency"].sum().reset_index()
matriz_completa = grid_completa.merge(conteo_real, on=["Topic", "post_year"], how="left").fillna({"Frequency": 0})

# 3. Métricas base y cálculo de la tasa de años inactivos
metricas_estabilidad = (
    tot_validos.groupby("Topic")
    .agg(
        volumen_total=("Frequency", "sum"),
        pico_maximo=("Frequency", "max"),
        frecuencia_media=("Frequency", "mean"),
        desvio_std=("Frequency", "std")
    )
    .reset_index()
)

porcentaje_ceros = (
    matriz_completa.groupby("Topic")["Frequency"]
    .apply(lambda x: (x == 0).mean())
    .reset_index(name="tasa_años_inactivos")
)

metricas_estabilidad = metricas_estabilidad.merge(porcentaje_ceros, on="Topic")

# 4. Nombres globales del modelo
info_global = topic_model.get_topic_info()
nombres_globales = {row["Topic"]: ", ".join(row["Representation"][:5]) for _, row in info_global.iterrows() if row["Topic"] != -1}
metricas_estabilidad["terminos_historicos"] = metricas_estabilidad["Topic"].map(nombres_globales)

metricas_estabilidad["desvio_std"] = metricas_estabilidad["desvio_std"].fillna(0)
metricas_estabilidad["coef_variacion"] = (metricas_estabilidad["desvio_std"] / metricas_estabilidad["frecuencia_media"]).round(2)

# 5. Clasificación automática con orden de prioridad corregido
umbral_cv = metricas_estabilidad["coef_variacion"].quantile(0.60)
umbral_cv_bajo = metricas_estabilidad["coef_variacion"].quantile(0.40)  
umbral_intermitencia = 0.25

condiciones = [
    # 1º Prioridad: Coyuntura Pura (Si el CV explota por las nubes, es coyuntura pura, ej: Pandemia o Aborto)
    (metricas_estabilidad["coef_variacion"] >= umbral_cv),
    
    # 2º Prioridad: Coyuntura Recurrente (Intermitente pero con variación moderada, ej: Rusia, Israel, Siria)
    (metricas_estabilidad["tasa_años_inactivos"] >= umbral_intermitencia) & (metricas_estabilidad["coef_variacion"] >= umbral_cv_bajo)
]

valores = [
    "Coyuntura",
    "Coyuntura Recurrente"
]

metricas_estabilidad["dinamica_temporal"] = np.select(condiciones, valores, default="Atemporal")

# 6. Time Decay automático
metricas_estabilidad["requiere_time_decay"] = metricas_estabilidad["dinamica_temporal"].isin(["Coyuntura", "Coyuntura Recurrente"])
metricas_estabilidad["factor_penalizacion_base"] = np.where(metricas_estabilidad["requiere_time_decay"], 0.5, 1.0)

metricas_estabilidad = metricas_estabilidad.sort_values(by="Topic").reset_index(drop=True)

# Visualización incluyendo la columna del decay (factor_penalizacion_base)
display(metricas_estabilidad[["Topic", "terminos_historicos", "dinamica_temporal", "coef_variacion", "tasa_años_inactivos", "factor_penalizacion_base"]])

,Topic,terminos_historicos,dinamica_temporal,coef_variacion,tasa_años_inactivos,factor_penalizacion_base
0,0,"político, política, discurso, junto, sociedad",Atemporal,0.39,0.066667,1.0
1,1,"noticia, sociedad, tecnológico, periodismo, te...",Coyuntura,0.82,0.200000,0.5
2,2,"aborto, abortar, anticonceptivo, embarazo, emb...",Coyuntura,1.70,0.200000,0.5
3,3,"pandemia, epidemia, vacunar, vacuna, vacunación",Coyuntura,1.25,0.266667,0.5
4,4,"papal, catolicismo, arzobispo, católica, católico",Coyuntura,0.90,0.200000,0.5
5,5,"arte, art, estética, artístico, artista",Atemporal,0.43,0.333333,1.0
6,6,"soviética, soviético, vladimir, rusia, ucrania",Coyuntura Recurrente,0.59,0.266667,0.5
7,7,"israel, netanyahu, israelí, palestina, palestino",Coyuntura Recurrente,0.46,0.333333,0.5
8,8,"cientificismo, astrología, universo, ciencia, ...",Coyuntura Recurrente,0.47,0.400000,0.5
9,9,"submarino, marino, océano, naval, marítimo",Atemporal,0.36,0.400000,1.0


In [82]:
import plotly.express as px

#minimo_notas = 20
MOSTRAR_POLITICA = False  # Cambiá a True si querés mostrar el Tópico 0

# 1. Filtramos ruido (-1) y aplicamos la regla del Tópico 0 según la variable MOSTRAR_POLITICA
if MOSTRAR_POLITICA:
    topicos_grafico = metricas_estabilidad[
        (metricas_estabilidad["Topic"] != -1) & 
        (metricas_estabilidad["volumen_total"] >= 5)
    ].copy()
else:
    topicos_grafico = metricas_estabilidad[
        (metricas_estabilidad["Topic"] != -1) & 
        (metricas_estabilidad["Topic"] != 0) & 
        (metricas_estabilidad["volumen_total"] >= 5)
    ].copy()

# 2. Imprimimos en consola para verificar los datos filtrados
print("--- DISTRIBUCIÓN DE TÓPICOS PARA EL GRÁFICO ---")
print(topicos_grafico[["Topic", "terminos_historicos", "dinamica_temporal", "coef_variacion"]].to_string())

# 3. Paleta de colores explícita para las tres dinámicas
colores_dinamica = {
    "Coyuntura": "#d62728",            # Rojo
    "Coyuntura Recurrente": "#ff7f0e",  # Naranja
    "Atemporal": "#2ca02c"             # Verde
}

# 4. Scatter Plot con el filtrado y colores correctos
fig_scatter = px.scatter(
    topicos_grafico,
    x="volumen_total",
    y="coef_variacion",
    color="dinamica_temporal",
    size="volumen_total",
    hover_name="terminos_historicos",
    text="Topic",
    color_discrete_map=colores_dinamica,
    title=f'<b>Cuadrante Editorial: Coyuntura, Recurrente y Estructura (Tópico 0 Visible: {MOSTRAR_POLITICA})</b>',
    labels={
        "volumen_total": "Volumen Total de Notas", 
        "coef_variacion": "Coeficiente de Variación (CV)",
        "dinamica_temporal": "Dinámica Editorial"
    },
    height=600
)

fig_scatter.update_traces(textposition='top center')

# Línea de referencia del umbral de CV
fig_scatter.add_hline(y=umbral_cv, line_dash="dash", line_color="gray", annotation_text="Umbral CV")

# Guardar y mostrar
fig_scatter.write_html(OUTPUT_DIR / "03A_cuadrante_editorial.html")
fig_scatter.show()

--- DISTRIBUCIÓN DE TÓPICOS PARA EL GRÁFICO ---
    Topic                                       terminos_historicos     dinamica_temporal  coef_variacion
1       1    noticia, sociedad, tecnológico, periodismo, tecnología             Coyuntura            0.82
2       2     aborto, abortar, anticonceptivo, embarazo, embarazado             Coyuntura            1.70
3       3           pandemia, epidemia, vacunar, vacuna, vacunación             Coyuntura            1.25
4       4         papal, catolicismo, arzobispo, católica, católico             Coyuntura            0.90
5       5                   arte, art, estética, artístico, artista             Atemporal            0.43
6       6            soviética, soviético, vladimir, rusia, ucrania  Coyuntura Recurrente            0.59
7       7          israel, netanyahu, israelí, palestina, palestino  Coyuntura Recurrente            0.46
8       8  cientificismo, astrología, universo, ciencia, científico  Coyuntura Recurrente            0.4